In [19]:
# ==========================================
# BACKTEST DU PIPELINE DE RECOMMANDATION
# ==========================================

import numpy as np
import pandas as pd

from sklearn.metrics.pairwise import cosine_similarity

print("Environnement du backtest chargé.")

Environnement du backtest chargé.


In [6]:
# ==========================================
# CHARGEMENT DE LA KNOWLEDGE BASE
# ==========================================

resolved_df = pd.read_excel(
    "../data/Inc_resolved by MAIT.xlsx"
)

escalated_df = pd.read_excel(
    "../data/Inc_registred by MAIT.xlsx"
)

resolved_df["source"] = "RESOLVED"
escalated_df["source"] = "ESCALATED"

knowledge_base = pd.concat(
    [
        resolved_df,
        escalated_df
    ],
    ignore_index=True
)

# Texte utilisé pour les embeddings
knowledge_base["text_for_embedding"] = (
    knowledge_base["Short Description"]
    .fillna("")
    .astype(str)
    + " "
    + knowledge_base["Description"]
    .fillna("")
    .astype(str)
)

print("Knowledge Base :", knowledge_base.shape)
print("Nombre de tickets :", len(knowledge_base))

C:\Users\LENOVO\PycharmProjects\DHL_AI_Incident_Assistant\.venv311\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Knowledge Base : (2243, 13)
Nombre de tickets : 2243


In [44]:
# ==========================================
# CLASSIFICATION DES SOLUTIONS
# ==========================================

import re

def classify_solution(solution):
    if pd.isna(solution):
        return "NON_ACTIONABLE"

    s = str(solution).strip().lower()

    if not s:
        return "NON_ACTIONABLE"

    # 1. ADMINISTRATIVE
    administrative_patterns = [
        "duplicate",
        "duplicated",
        "close the ticket",
        "ticket to avoid",
        "no response from user",
        "regional ticket opened",
        "new ticket has been raised",
        "a new ticket has been raised",
        "ticket has been raised",
        "ticket opened",
        "data provided",
        "information provided",
        "wrongly routed",
        "cancelled",
        "cancelled by",
    ]

    if any(pattern in s for pattern in administrative_patterns):
        return "ADMINISTRATIVE"

    # 2. NON_ACTIONABLE
    non_actionable_patterns = [
        "issue under provider investigations",
        "provider is working on it",
        "provider is working on the issue",
        "no approval is provided",
        "required information needed to resolve",
        "required information was not provided",
        "please ask performance to provide",
        "please reach out to",
        "please contact",
        "please request access from",
        "not our responsibility",
        "no issue from our side",
        "waiting for",
        "waiting on",
        "pending",
        "under investigation",
        "changes ongoing",
        "will be changed",
        "will be configured",
        "will be installed",
    ]

    if any(pattern in s for pattern in non_actionable_patterns):
        return "NON_ACTIONABLE"

    # 3. GENERIC
    generic_solutions = {
        "resolved",
        "resolved.",
        "resolved,",
        "solved",
        "solved.",
        "done",
        "done.",
        "fixed",
        "fixed.",
        "closed",
        "closed.",
        "completed",
        "completed.",
        "granted",
        "access granted",
        "access granted.",
        "necessary done",
        "necessary done.",
        "necessaey done.",
        "necessary changes are made.",
        "the necessary changes are made.",
        "change done",
        "change done.",
        "fixed1",
    }

    if s in generic_solutions:
        return "GENERIC"

    if re.match(r"^(resolved|fixed)\s+inc[\w-]+", s):
        return "GENERIC"

    # 4. ACTIONABLE
    actionable_patterns = [
        "installed",
        "installation done",
        "created",
        "was created",
        "has been created",
        "configured",
        "configuration done",
        "updated",
        "has been updated",
        "was updated",
        "changed",
        "has been changed",
        "was changed",
        "replaced",
        "has been replaced",
        "removed",
        "has been removed",
        "reset",
        "password generated",
        "password has been sent",
        "password was sent",
        "unlocked",
        "unlock account",
        "access granted",
        "access provided",
        "granted the",
        "enabled",
        "disabled",
        "recreated",
        "fixed by",
        "released",
        "session released",
        "restored",
        "activated",
        "completed successfully",
        "request completed",
        "changes complete",
        "changes were made",
        "changes are made",
        "set as requested",
        "printer has been created",
        "printers were recreated",
        "rebooted",
        "reconfigured",
        "configured a new",
        "synced",
        "provided the access",
        "has been granted",
    ]

    if any(pattern in s for pattern in actionable_patterns):
        return "ACTIONABLE"

    # 5. Par défaut
    return "NON_ACTIONABLE"

In [45]:
knowledge_base["solution_type"] = (
    knowledge_base["Solution"]
    .apply(classify_solution)
)

print(
    knowledge_base["solution_type"]
    .value_counts()
)

solution_type
GENERIC           1401
NON_ACTIONABLE     587
ACTIONABLE         213
ADMINISTRATIVE      42
Name: count, dtype: int64


In [9]:
import numpy as np

print("Nombre de lignes knowledge_base :", len(knowledge_base))

print("Shape ticket_embeddings :", embeddings.shape)
print("Shape solution_embeddings :", solution_embeddings.shape)

print()
print("Vérification de l'alignement :")

assert len(knowledge_base) == embeddings.shape[0], \
    "ERREUR : ticket_embeddings et knowledge_base ne sont pas alignés"

assert len(knowledge_base) == solution_embeddings.shape[0], \
    "ERREUR : solution_embeddings et knowledge_base ne sont pas alignés"

print("OK : les 3 éléments sont alignés.")

Nombre de lignes knowledge_base : 2243
Shape ticket_embeddings : (2243, 384)
Shape solution_embeddings : (2243, 384)

Vérification de l'alignement :
OK : les 3 éléments sont alignés.


In [4]:
import numpy as np
import pandas as pd
from pathlib import Path

# Charger les embeddings pré-calculés
embeddings = np.load("../data/ticket_embeddings.npy")
solution_embeddings = np.load("../data/solution_embeddings.npy")

print("Ticket embeddings :", embeddings.shape)
print("Solution embeddings :", solution_embeddings.shape)

Ticket embeddings : (2243, 384)
Solution embeddings : (2243, 384)


In [10]:
# ==========================================
# CHARGEMENT DES EMBEDDINGS PRÉ-CALCULÉS
# ==========================================

embeddings = np.load("../data/ticket_embeddings.npy")
solution_embeddings = np.load("../data/solution_embeddings.npy")

print("Ticket embeddings :", embeddings.shape)
print("Solution embeddings :", solution_embeddings.shape)

assert len(knowledge_base) == embeddings.shape[0], \
    "ERREUR : ticket_embeddings et knowledge_base ne sont pas alignés"

assert len(knowledge_base) == solution_embeddings.shape[0], \
    "ERREUR : solution_embeddings et knowledge_base ne sont pas alignés"

print("OK : les 3 éléments sont alignés.")

Ticket embeddings : (2243, 384)
Solution embeddings : (2243, 384)
OK : les 3 éléments sont alignés.


In [55]:
# ==========================================
# FONCTIONS DU PIPELINE UTILISÉES PAR LE BACKTEST
# ==========================================

from sklearn.cluster import AgglomerativeClustering


# ==========================================
# 3. CLUSTERING DES SOLUTIONS
# ==========================================

def cluster_similar_solutions(
    solution_similarity_matrix,
    threshold=0.75
):
    """
    Regroupe les solutions historiquement similaires.

    threshold = seuil minimal de similarité entre solutions.

    Cas particuliers :
    - 0 solution  -> aucun cluster
    - 1 solution  -> un cluster unique
    - >= 2 solutions -> AgglomerativeClustering
    """

    # Aucun candidat
    if solution_similarity_matrix.empty:
        return pd.DataFrame(
            columns=["Number", "Cluster"]
        )

    # Un seul candidat :
    # impossible d'utiliser AgglomerativeClustering,
    # mais la solution constitue naturellement un cluster unique.
    if len(solution_similarity_matrix) == 1:

        return pd.DataFrame({
            "Number": solution_similarity_matrix.index,
            "Cluster": [0]
        })

    # Cas normal : au moins 2 solutions
    distance_matrix = 1 - solution_similarity_matrix.values

    clustering = AgglomerativeClustering(
        n_clusters=None,
        metric="precomputed",
        linkage="average",
        distance_threshold=1 - threshold
    )

    labels = clustering.fit_predict(distance_matrix)

    clusters = pd.DataFrame({
        "Number": solution_similarity_matrix.index,
        "Cluster": labels
    })

    return clusters


# ==========================================
# 5. SCORING DES CLUSTERS
# ==========================================

def score_solution_clusters(
    actionable,
    clusters,
    solution_similarity_matrix
):
    """
    Calcule un score pour chaque cluster de solutions.

    Le score prend en compte :
    - pertinence des tickets
    - cohérence des solutions
    - nombre de tickets dans le cluster
    """

    cluster_results = []

    for cluster_id in sorted(clusters["Cluster"].unique()):

        cluster_numbers = clusters.loc[
            clusters["Cluster"] == cluster_id,
            "Number"
        ].tolist()

        cluster_data = actionable[
            actionable["Number"].isin(cluster_numbers)
        ].copy()

        if cluster_data.empty:
            continue

        # Pertinence moyenne des tickets
        ticket_similarity_mean = (
            cluster_data["Ticket_Similarity"].mean()
        )

        # Meilleure pertinence du cluster
        ticket_similarity_max = (
            cluster_data["Ticket_Similarity"].max()
        )

        # Cohérence entre les solutions
        valid_tickets = cluster_data["Number"].tolist()

        if len(valid_tickets) > 1:

            matrix = solution_similarity_matrix.loc[
                valid_tickets,
                valid_tickets
            ].values

            upper_values = matrix[
                np.triu_indices(
                    len(matrix),
                    k=1
                )
            ]

            solution_coherence = upper_values.mean()

        else:
            # Une seule solution = pas de conflit interne
            solution_coherence = 0.5

        cluster_results.append({
            "Cluster": cluster_id,
            "Number_of_Tickets": len(cluster_data),
            "Ticket_Similarity_Mean": ticket_similarity_mean,
            "Ticket_Similarity_Max": ticket_similarity_max,
            "Solution_Coherence": solution_coherence
        })

    return pd.DataFrame(cluster_results)


# ==========================================
# 6. CLASSEMENT DES CLUSTERS
# ==========================================

def rank_solution_clusters(cluster_scores):

    if cluster_scores.empty:
        return cluster_scores.copy()

    scores = cluster_scores.copy()

    # Nombre maximal de tickets dans un cluster
    max_support = scores["Number_of_Tickets"].max()

    # Pertinence
    scores["Relevance_Score"] = (
        scores["Ticket_Similarity_Mean"]
    )

    # Cohérence des solutions
    scores["Coherence_Score"] = (
        scores["Solution_Coherence"]
    )

    # Support historique
    scores["Support_Score"] = (
        scores["Number_of_Tickets"] / max_support
    )

    # Score final
    scores["Cluster_Final_Score"] = (
        0.5 * scores["Relevance_Score"]
        + 0.3 * scores["Coherence_Score"]
        + 0.2 * scores["Support_Score"]
    )

    return scores.sort_values(
        "Cluster_Final_Score",
        ascending=False
    ).reset_index(drop=True)

In [57]:
# ==========================================
# BACKTEST D'UN TICKET - VERSION FINALE
# ==========================================

def run_single_backtest(
    test_ticket_number,
    top_k=50,
    cluster_threshold=0.75
):
    """
    Évalue le pipeline sur un ticket historique.

    Le ticket testé est retiré de la knowledge base
    afin d'éviter toute fuite de données.

    Les embeddings pré-calculés sont réutilisés.

    Parameters
    ----------
    test_ticket_number : str
        Numéro du ticket à tester.

    top_k : int
        Nombre de tickets similaires récupérés.

    cluster_threshold : float
        Seuil de similarité pour regrouper les
        solutions dans les clusters.
    """

    # ======================================
    # 1. LOCALISER LE TICKET DE TEST
    # ======================================

    matches = knowledge_base.index[
        knowledge_base["Number"]
        .astype(str)
        .eq(str(test_ticket_number))
    ].tolist()

    if len(matches) == 0:
        raise ValueError(
            f"Ticket introuvable : {test_ticket_number}"
        )

    if len(matches) > 1:
        raise ValueError(
            f"Plusieurs tickets trouvés pour : "
            f"{test_ticket_number}"
        )

    test_index = matches[0]

    test_row = knowledge_base.loc[test_index]

    true_solution = str(
        test_row["Solution"]
    )

    # ======================================
    # 2. RETIRER LE TICKET DE TEST
    # ======================================

    local_indices = knowledge_base.index[
        knowledge_base.index != test_index
    ].to_numpy()

    local_kb = knowledge_base.loc[
        local_indices
    ].copy()

    local_ticket_embeddings = embeddings[
        local_indices
    ]

    # ======================================
    # 3. EMBEDDING DU TICKET TEST
    # ======================================

    query_embedding = embeddings[
        test_index
    ]

    # ======================================
    # 4. RECHERCHE DES TICKETS SIMILAIRES
    # ======================================

    ticket_similarities = cosine_similarity(
        local_ticket_embeddings,
        query_embedding.reshape(1, -1)
    ).flatten()

    top_k = min(
        top_k,
        len(local_kb)
    )

    top_positions = np.argsort(
        ticket_similarities
    )[::-1][:top_k]

    all_candidates = local_kb.iloc[
        top_positions
    ].copy()

    all_candidates["Ticket_Similarity"] = (
        ticket_similarities[top_positions]
    )

    # Trier explicitement les candidats
    all_candidates = all_candidates.sort_values(
        "Ticket_Similarity",
        ascending=False
    )

    # ======================================
    # 5. FILTRER LES SOLUTIONS ACTIONABLE
    # ======================================

    actionable = all_candidates[
        all_candidates["solution_type"]
        == "ACTIONABLE"
    ].copy()

    # --------------------------------------
    # Aucun candidat ACTIONABLE
    # --------------------------------------

    if actionable.empty:

        return {
            "status": "NO_ACTIONABLE_SOLUTION",

            "Test_Ticket":
                test_ticket_number,

            "True_Solution":
                true_solution,

            "All_Candidates":
                all_candidates,

            "Candidates":
                actionable,

            "Confidence_Level":
                "LOW",

            "Confidence_Score":
                0.0
        }

    # ======================================
    # 6. EMBEDDINGS DES SOLUTIONS
    # ======================================

    candidate_original_indices = (
        actionable.index.to_numpy()
    )

    candidate_solution_embeddings = (
        solution_embeddings[
            candidate_original_indices
        ]
    )

    # ======================================
    # 7. SIMILARITÉ ENTRE SOLUTIONS
    # ======================================

    # --------------------------------------
    # Cas : un seul candidat
    # --------------------------------------

    if len(actionable) == 1:

        solution_similarity_matrix = pd.DataFrame(
            [[1.0]],
            index=actionable["Number"],
            columns=actionable["Number"]
        )

    # --------------------------------------
    # Cas : plusieurs candidats
    # --------------------------------------

    else:

        solution_matrix = cosine_similarity(
            candidate_solution_embeddings
        )

        solution_similarity_matrix = pd.DataFrame(
            solution_matrix,
            index=actionable["Number"],
            columns=actionable["Number"]
        )

    # ======================================
    # 8. CLUSTERING DES SOLUTIONS
    # ======================================

    clusters = cluster_similar_solutions(
        solution_similarity_matrix,
        threshold=cluster_threshold
    )

    # ======================================
    # 9. SCORE DES CLUSTERS
    # ======================================

    cluster_scores = score_solution_clusters(
        actionable,
        clusters,
        solution_similarity_matrix
    )

    # --------------------------------------
    # Aucun cluster valide
    # --------------------------------------

    if cluster_scores.empty:

        return {
            "status": "NO_VALID_CLUSTER",

            "Test_Ticket":
                test_ticket_number,

            "True_Solution":
                true_solution,

            "All_Candidates":
                all_candidates,

            "Candidates":
                actionable,

            "Confidence_Level":
                "LOW",

            "Confidence_Score":
                0.0
        }

    # ======================================
    # 10. CLASSEMENT DES CLUSTERS
    # ======================================

    ranked_clusters = rank_solution_clusters(
        cluster_scores
    )

    # ======================================
    # 11. REPRÉSENTANT DE CHAQUE CLUSTER
    # ======================================

    representatives = []

    for cluster_id in sorted(
        clusters["Cluster"].unique()
    ):

        cluster_numbers = clusters.loc[
            clusters["Cluster"] == cluster_id,
            "Number"
        ].tolist()

        cluster_data = actionable[
            actionable["Number"].isin(
                cluster_numbers
            )
        ].copy()

        # ----------------------------------
        # Cluster avec un seul ticket
        # ----------------------------------

        if len(cluster_numbers) == 1:

            row = cluster_data.iloc[0]

            representatives.append({

                "Cluster":
                    cluster_id,

                "Number":
                    row["Number"],

                "Solution":
                    row["Solution"],

                "Ticket_Similarity":
                    row["Ticket_Similarity"],

                # Un singleton ne permet pas
                # de mesurer une cohérence réelle
                "Internal_Similarity":
                    0.5
            })

        # ----------------------------------
        # Cluster avec plusieurs tickets
        # ----------------------------------

        else:

            matrix = solution_similarity_matrix.loc[
                cluster_numbers,
                cluster_numbers
            ]

            mean_similarities = {}

            for ticket_number in cluster_numbers:

                others = [
                    number
                    for number in cluster_numbers
                    if number != ticket_number
                ]

                mean_similarities[
                    ticket_number
                ] = matrix.loc[
                    ticket_number,
                    others
                ].mean()

            # ------------------------------
            # Solution centrale du cluster
            # ------------------------------

            representative_ticket = max(
                mean_similarities,
                key=mean_similarities.get
            )

            row = cluster_data[
                cluster_data["Number"]
                == representative_ticket
            ].iloc[0]

            representatives.append({

                "Cluster":
                    cluster_id,

                "Number":
                    representative_ticket,

                "Solution":
                    row["Solution"],

                "Ticket_Similarity":
                    row["Ticket_Similarity"],

                "Internal_Similarity":
                    mean_similarities[
                        representative_ticket
                    ]
            })

    representatives = pd.DataFrame(
        representatives
    )

    # ======================================
    # 12. SÉLECTION DU MEILLEUR CLUSTER
    # ======================================

    best_cluster = int(
        ranked_clusters.iloc[0]["Cluster"]
    )

    best_cluster_data = ranked_clusters.iloc[0]

    representative = representatives[
        representatives["Cluster"]
        == best_cluster
    ].iloc[0]

    representative_ticket = (
        representative["Number"]
    )

    representative_solution = str(
        representative["Solution"]
    )

    # ======================================
    # 13. COMPARAISON AVEC VRAIE SOLUTION
    # ======================================

    true_solution_embedding = (
        solution_embeddings[test_index]
    )

    # L'index du représentant est déjà
    # présent dans actionable

    representative_index = actionable[
        actionable["Number"].astype(str)
        == str(representative_ticket)
    ].index[0]

    representative_solution_embedding = (
        solution_embeddings[
            representative_index
        ]
    )

    solution_similarity = cosine_similarity(
        [true_solution_embedding],
        [representative_solution_embedding]
    )[0][0]

    # ======================================
    # 14. CALCUL DU CONFIDENCE SCORE
    # ======================================

    ticket_similarity = float(
        representative["Ticket_Similarity"]
    )

    cluster_score = float(
        best_cluster_data[
            "Cluster_Final_Score"
        ]
    )

    internal_similarity = float(
        representative[
            "Internal_Similarity"
        ]
    )

    number_of_tickets = int(
        best_cluster_data[
            "Number_of_Tickets"
        ]
    )

    # --------------------------------------
    # Score de confiance
    # --------------------------------------

    confidence_score = (
        0.50 * ticket_similarity
        + 0.30 * internal_similarity
        + 0.20 * cluster_score
    )

    # --------------------------------------
    # Pénalité pour cluster singleton
    # --------------------------------------

    if number_of_tickets == 1:

        confidence_score *= 0.80

    # ======================================
    # 15. NIVEAU DE CONFIANCE
    # ======================================

    if (
        confidence_score >= 0.80
        and number_of_tickets >= 2
    ):

        confidence_level = "HIGH"

    elif confidence_score >= 0.60:

        confidence_level = "MEDIUM"

    else:

        confidence_level = "LOW"

    # ======================================
    # 16. RÉSULTAT FINAL
    # ======================================

    return {

        "status":
            "SUCCESS",

        "Test_Ticket":
            test_ticket_number,

        "True_Solution":
            true_solution,

        # ----------------------------------
        # Cluster gagnant
        # ----------------------------------

        "Best_Cluster":
            best_cluster,

        "Number_of_Tickets":
            number_of_tickets,

        # ----------------------------------
        # Solution recommandée
        # ----------------------------------

        "Representative_Ticket":
            representative_ticket,

        "Representative_Solution":
            representative_solution,

        # ----------------------------------
        # Scores
        # ----------------------------------

        "Ticket_Similarity":
            ticket_similarity,

        "Internal_Similarity":
            internal_similarity,

        "Cluster_Score":
            cluster_score,

        "Solution_Similarity":
            float(solution_similarity),

        # ----------------------------------
        # Confiance
        # ----------------------------------

        "Confidence_Score":
            confidence_score,

        "Confidence_Level":
            confidence_level,

        # ----------------------------------
        # Données de diagnostic
        # ----------------------------------

        "All_Candidates":
            all_candidates,

        "Candidates":
            actionable,

        "Ranked_Clusters":
            ranked_clusters,

        "Representatives":
            representatives
    }

In [23]:
print(
    "Test ticket présent dans les candidats :",
    result["Candidates"]["Number"]
        .astype(str)
        .eq("INC61302608")
        .any()
)

Test ticket présent dans les candidats : False


In [60]:
# ==========================================
# DIAGNOSTIC DÉTAILLÉ DU BACKTEST
# ==========================================

def diagnose_backtest(
    test_ticket_number,
    top_k=10,
    cluster_threshold=0.75
):

    print("=" * 80)
    print(f"DIAGNOSTIC : {test_ticket_number}")
    print("=" * 80)

    # --------------------------------------
    # 1. EXÉCUTION DU BACKTEST
    # --------------------------------------

    result = run_single_backtest(
        test_ticket_number,
        top_k=top_k,
        cluster_threshold=cluster_threshold
    )

    print("\nStatus :", result["status"])

    # --------------------------------------
    # 2. INFORMATIONS DU TICKET TEST
    # --------------------------------------

    test_row = knowledge_base[
        knowledge_base["Number"].astype(str)
        == str(test_ticket_number)
    ].iloc[0]

    print("\n--- TICKET TESTÉ ---")

    print("Number :", test_ticket_number)

    print(
        "Short Description :",
        test_row["Short Description"]
    )

    print(
        "Vraie solution :",
        test_row["Solution"]
    )

    # --------------------------------------
    # 3. CANDIDATS
    # --------------------------------------

    candidates = result.get("Candidates")

    if candidates is not None and not candidates.empty:

        candidates = candidates.copy()

        if "Ticket_Similarity" in candidates.columns:

            candidates = candidates.sort_values(
                "Ticket_Similarity",
                ascending=False
            )

        print("\n--- CANDIDATS ---")

        columns_to_display = [
            "Number",
            "Short Description",
            "Ticket_Similarity",
            "Solution",
            "solution_type"
        ]

        available_columns = [
            col for col in columns_to_display
            if col in candidates.columns
        ]

        display(
            candidates[
                available_columns
            ].head(top_k)
        )

    else:

        print("\nAucun candidat trouvé.")

    # --------------------------------------
    # 4. ARRÊT SI PAS DE SUCCÈS
    # --------------------------------------

    if result["status"] != "SUCCESS":

        print("\n" + "=" * 80)
        print("BACKTEST NON COMPLET")
        print("=" * 80)

        print(
            "\nLe pipeline n'a pas pu produire "
            "une recommandation finale."
        )

        print(
            "\nRaison :",
            result["status"]
        )

        if result["status"] == "NO_ACTIONABLE_SOLUTION":

            print(
                "\nAucune solution ACTIONABLE "
                "n'a été trouvée parmi les "
                f"{top_k} meilleurs tickets similaires."
            )

            if candidates is not None:

                print("\nRépartition des types :")

                print(
                    candidates["solution_type"]
                    .value_counts()
                )

        elif result["status"] == "NO_VALID_CLUSTER":

            print(
                "\nDes candidats ACTIONABLE ont été trouvés, "
                "mais aucun cluster valide n'a pu être créé."
            )

        return result

    # --------------------------------------
    # 5. CLUSTERS
    # --------------------------------------

    ranked_clusters = result[
        "Ranked_Clusters"
    ].copy()

    print("\n--- CLUSTERS CLASSÉS ---")

    display(ranked_clusters)

    # --------------------------------------
    # 6. REPRÉSENTANTS
    # --------------------------------------

    representatives = result[
        "Representatives"
    ].copy()

    print("\n--- REPRÉSENTANTS DES CLUSTERS ---")

    display(representatives)

    # --------------------------------------
    # 7. CLUSTER GAGNANT
    # --------------------------------------

    best_cluster = result[
        "Best_Cluster"
    ]

    representative_ticket = result[
        "Representative_Ticket"
    ]

    representative_solution = result[
        "Representative_Solution"
    ]

    print("\n--- CLUSTER GAGNANT ---")

    print(
        "Cluster :",
        best_cluster
    )

    print(
        "Ticket représentant :",
        representative_ticket
    )

    print(
        "Ticket similarity :",
        result["Ticket_Similarity"]
    )

    print(
        "Internal similarity :",
        result["Internal_Similarity"]
    )

    print(
        "Cluster score :",
        result["Cluster_Score"]
    )

    print(
        "Solution similarity :",
        result["Solution_Similarity"]
    )

    # --------------------------------------
    # 8. SOLUTION REPRÉSENTATIVE
    # --------------------------------------

    print("\n--- SOLUTION REPRÉSENTATIVE ---")

    print(representative_solution)

    # --------------------------------------
    # 9. COMPARAISON FINALE
    # --------------------------------------

    print("\n--- COMPARAISON ---")

    print("VRAIE SOLUTION :")
    print(test_row["Solution"])

    print("\nSOLUTION RETOURNÉE :")
    print(representative_solution)

    print(
        "\nSimilarité entre les deux :",
        result["Solution_Similarity"]
    )

    return result

In [61]:
diagnose_backtest(
    "INC67571325",
    top_k=20
)

DIAGNOSTIC : INC67571325

Status : SUCCESS

--- TICKET TESTÉ ---
Number : INC67571325
Short Description : MA - EXP - RESET PASSWORD LDAP : zelhadda
Vraie solution : Hello Oussama,

Please note the new password : Focus@147258369@

--- CANDIDATS ---


,Number,Short Description,Ticket_Similarity,Solution,solution_type
713,INC67156306,MA - EXP - Connection problems with Outlook an...,0.762826,"Invistigation done.\nUser disabled by PIAM , n...",ACTIONABLE



--- CLUSTERS CLASSÉS ---


,Cluster,Number_of_Tickets,Ticket_Similarity_Mean,Ticket_Similarity_Max,Solution_Coherence,Relevance_Score,Coherence_Score,Support_Score,Cluster_Final_Score
0,0,1,0.762826,0.762826,0.5,0.762826,0.5,1.0,0.731413



--- REPRÉSENTANTS DES CLUSTERS ---


,Cluster,Number,Solution,Ticket_Similarity,Internal_Similarity
0,0,INC67156306,"Invistigation done.\nUser disabled by PIAM , n...",0.762826,0.5



--- CLUSTER GAGNANT ---
Cluster : 0
Ticket représentant : INC67156306
Ticket similarity : 0.7628262042999268
Internal similarity : 0.5
Cluster score : 0.7314131021499635
Solution similarity : 0.3255992531776428

--- SOLUTION REPRÉSENTATIVE ---
Invistigation done.
User disabled by PIAM , need to be extanded  by HR team.

--- COMPARAISON ---
VRAIE SOLUTION :
Hello Oussama,

Please note the new password : Focus@147258369@

SOLUTION RETOURNÉE :
Invistigation done.
User disabled by PIAM , need to be extanded  by HR team.

Similarité entre les deux : 0.3255992531776428


{'status': 'SUCCESS',
 'Test_Ticket': 'INC67571325',
 'True_Solution': 'Hello Oussama,\n\nPlease note the new password : Focus@147258369@',
 'Best_Cluster': 0,
 'Number_of_Tickets': 1,
 'Representative_Ticket': 'INC67156306',
 'Representative_Solution': 'Invistigation done.\nUser disabled by PIAM , need to be extanded  by HR team.',
 'Ticket_Similarity': 0.7628262042999268,
 'Internal_Similarity': 0.5,
 'Cluster_Score': 0.7314131021499635,
 'Solution_Similarity': 0.3255992531776428,
 'Confidence_Score': 0.5421565780639649,
 'Confidence_Level': 'LOW',
 'All_Candidates':            Number              Opened Target Resolution Time (GMT)  \
 707   INC69279844 2026-03-31 09:29:07          2026-04-07 09:29:07   
 717   INC69280550 2026-03-31 09:59:44          2026-04-07 09:59:44   
 703   INC69279873 2026-03-31 09:30:09          2026-04-07 09:30:09   
 706   INC69282830 2026-03-31 11:38:11          2026-04-07 11:38:11   
 715   INC69282812 2026-03-31 11:37:22          2026-04-07 11:37:22   

In [54]:
diagnose_backtest(
    "INC72044971",
    top_k=20
)

DIAGNOSTIC : INC72044971

Status : SUCCESS

--- TICKET TESTÉ ---
Number : INC72044971
Short Description : MA - EXP - Correction of Phone Number on Invoices and Dunning Letters
Vraie solution : Fixed by Chetan

--- CANDIDATS ---


,Number,Short Description,Ticket_Similarity,Solution,solution_type
1989,INC61058067,Telephony down,0.662499,"The variable has been released. Next time, in ...",ACTIONABLE



--- CLUSTERS CLASSÉS ---


,Cluster,Number_of_Tickets,Ticket_Similarity_Mean,Ticket_Similarity_Max,Solution_Coherence,Relevance_Score,Coherence_Score,Support_Score,Cluster_Final_Score
0,0,1,0.662499,0.662499,1.0,0.662499,1.0,1.0,0.831249



--- REPRÉSENTANTS DES CLUSTERS ---


,Cluster,Number,Solution,Ticket_Similarity,Internal_Similarity
0,0,INC61058067,"The variable has been released. Next time, in ...",0.662499,1.0



--- CLUSTER GAGNANT ---
Cluster : 0
Ticket représentant : INC61058067
Ticket similarity : 0.6624986
Internal similarity : 1.0
Cluster score : 0.8312492966651917
Solution similarity : 0.1547108

--- SOLUTION REPRÉSENTATIVE ---
The variable has been released. Next time, in such case that the call centre is unavailable please call Emergency Incident Manager and ask for E4 ticket to resolve the issue ASAP.

--- COMPARAISON ---
VRAIE SOLUTION :
Fixed by Chetan

SOLUTION RETOURNÉE :
The variable has been released. Next time, in such case that the call centre is unavailable please call Emergency Incident Manager and ask for E4 ticket to resolve the issue ASAP.

Similarité entre les deux : 0.1547108


{'status': 'SUCCESS',
 'Test_Ticket': 'INC72044971',
 'True_Solution': 'Fixed by Chetan',
 'Best_Cluster': 0,
 'Representative_Ticket': 'INC61058067',
 'Representative_Solution': 'The variable has been released. Next time, in such case that the call centre is unavailable please call Emergency Incident Manager and ask for E4 ticket to resolve the issue ASAP.',
 'Ticket_Similarity': np.float32(0.6624986),
 'Internal_Similarity': np.float64(1.0),
 'Cluster_Score': np.float64(0.8312492966651917),
 'Solution_Similarity': np.float32(0.1547108),
 'Candidates':            Number              Opened Target Resolution Time (GMT)  \
 1989  INC61058067 2024-09-28 09:56:55          2024-09-30 16:00:00   
 
              Informed at Incident Status         Caller  \
 1989 2024-09-28 10:59:04          Closed  Nabil Mahmoud   
 
                        Assigned Group     Assignee Short Description  \
 1989  AP-SS.ContactCenter.CSTelephony  Tomas Chlum    Telephony down   
 
                           

In [62]:
diagnose_backtest("INC61302608")

DIAGNOSTIC : INC61302608

Status : SUCCESS

--- TICKET TESTÉ ---
Number : INC61302608
Short Description : Add printers on THOR app
Vraie solution : done all mentioned printers installed on servers.

MYKULLSPC000578
device for MATNGKEN03: beh:/1/0/60/socket://23.82.61.234:9100
MATNGKEN03 accepting requests since Thu 17 Oct 2024 08:10:21 AM GMT
printer MATNGKEN03 is idle.  enabled since Thu 17 Oct 2024 08:10:21 AM GMT
device for MATNGKEN04: beh:/1/0/60/socket://23.82.61.235:9100
MATNGKEN04 accepting requests since Thu 17 Oct 2024 08:10:21 AM GMT
printer MATNGKEN04 is idle.  enabled since Thu 17 Oct 2024 08:10:21 AM GMT
device for MARBARAK03: beh:/1/0/60/socket://23.82.50.234:9100
MARBARAK03 accepting requests since Thu 17 Oct 2024 08:10:21 AM GMT
printer MARBARAK03 is idle.  enabled since Thu 17 Oct 2024 08:10:21 AM GMT
device for MARBARAK04: beh:/1/0/60/socket://23.82.50.235:9100
MARBARAK04 accepting requests since Thu 17 Oct 2024 08:10:21 AM GMT
printer MARBARAK04 is idle.  enabled sin

,Number,Short Description,Ticket_Similarity,Solution,solution_type
1981,INC61302569,Add printers on GCA7 app,0.901350,"Completed, printers installed on demanded serv...",ACTIONABLE
1962,INC66170396,Add printers on GCA7 app,0.769743,Printer has been created successfully\n,ACTIONABLE
1956,INC69349039,Add printers on GCA7,0.769594,"Hi, Printer is added to requested servers. Tha...",ACTIONABLE
1827,INC65410555,Request to change printer model for GCA printe...,0.687134,As per the request printer driver updated \n\n...,ACTIONABLE



--- CLUSTERS CLASSÉS ---


,Cluster,Number_of_Tickets,Ticket_Similarity_Mean,Ticket_Similarity_Max,Solution_Coherence,Relevance_Score,Coherence_Score,Support_Score,Cluster_Final_Score
0,0,2,0.835472,0.901350,0.810523,0.835472,0.810523,1.0,0.860893
1,1,1,0.769743,0.769743,0.500000,0.769743,0.500000,0.5,0.634871
2,2,1,0.687134,0.687134,0.500000,0.687134,0.500000,0.5,0.593567



--- REPRÉSENTANTS DES CLUSTERS ---


,Cluster,Number,Solution,Ticket_Similarity,Internal_Similarity
0,0,INC61302569,"Completed, printers installed on demanded serv...",0.901350,0.810523
1,1,INC66170396,Printer has been created successfully\n,0.769743,0.500000
2,2,INC65410555,As per the request printer driver updated \n\n...,0.687134,0.500000



--- CLUSTER GAGNANT ---
Cluster : 0
Ticket représentant : INC61302569
Ticket similarity : 0.9013497829437256
Internal similarity : 0.8105230927467346
Cluster score : 0.8608929216861725
Solution similarity : 0.8952406644821167

--- SOLUTION REPRÉSENTATIVE ---
Completed, printers installed on demanded servers:
MAKENPR4
----------
device for MAKENPR4: beh:/1/0/60/socket://23.82.61.234:9100
MAKENPR4 accepting requests since Wed 16 Oct 2024 10:33:21 PM GMT
printer MAKENPR4 is idle.  enabled since Wed 16 Oct 2024 10:33:21 PM GMT
MAKENPR3
----------
device for MAKENPR3: beh:/1/0/60/socket://23.82.61.235:9100
MAKENPR3 accepting requests since Wed 16 Oct 2024 10:33:21 PM GMT
printer MAKENPR3 is idle.  enabled since Wed 16 Oct 2024 10:33:21 PM GMT
MARAKPR4
----------
device for MARAKPR4: beh:/1/0/60/socket://23.82.50.234:9100
MARAKPR4 accepting requests since Wed 16 Oct 2024 10:33:21 PM GMT
printer MARAKPR4 is idle.  enabled since Wed 16 Oct 2024 10:33:21 PM GMT
MARAKPR3
----------
device for M

{'status': 'SUCCESS',
 'Test_Ticket': 'INC61302608',
 'True_Solution': 'done all mentioned printers installed on servers.\n\nMYKULLSPC000578\ndevice for MATNGKEN03: beh:/1/0/60/socket://23.82.61.234:9100\nMATNGKEN03 accepting requests since Thu 17 Oct 2024 08:10:21 AM GMT\nprinter MATNGKEN03 is idle.  enabled since Thu 17 Oct 2024 08:10:21 AM GMT\ndevice for MATNGKEN04: beh:/1/0/60/socket://23.82.61.235:9100\nMATNGKEN04 accepting requests since Thu 17 Oct 2024 08:10:21 AM GMT\nprinter MATNGKEN04 is idle.  enabled since Thu 17 Oct 2024 08:10:21 AM GMT\ndevice for MARBARAK03: beh:/1/0/60/socket://23.82.50.234:9100\nMARBARAK03 accepting requests since Thu 17 Oct 2024 08:10:21 AM GMT\nprinter MARBARAK03 is idle.  enabled since Thu 17 Oct 2024 08:10:21 AM GMT\ndevice for MARBARAK04: beh:/1/0/60/socket://23.82.50.235:9100\nMARBARAK04 accepting requests since Thu 17 Oct 2024 08:10:21 AM GMT\nprinter MARBARAK04 is idle.  enabled since Thu 17 Oct 2024 08:10:21 AM GMT\ndevice for MARBAAGA03: beh

In [50]:
diagnose_backtest("INC67571325")

DIAGNOSTIC : INC67571325

--- TICKET TESTÉ ---
Number : INC67571325
Short Description : MA - EXP - RESET PASSWORD LDAP : zelhadda
Vraie solution : Hello Oussama,

Please note the new password : Focus@147258369@

--- TOP CANDIDATS ---


,Number,Short Description,Ticket_Similarity,Solution,solution_type
707,INC69279844,MA - EXP - NEW ACCOUNT OCNT LDAP josym1,0.844316,Resolved,GENERIC
717,INC69280550,MA - EXP - NEW ACCOUNT DCE LDAP josym1,0.842690,Access Granted,GENERIC
703,INC69279873,MA - EXP - NEW ACCOUNT NMIV LDAP josym1,0.841746,resolved,GENERIC
706,INC69282830,MA - EXP - NEW ACCOUNT PRIM LDAP josym1,0.838885,"Resolved , ticket ouvert INC69386410",NON_ACTIONABLE
715,INC69282812,MA - EXP - NEW ACCOUNT HMS LDAP josym1,0.831485,Resolved,GENERIC
1492,INC68501709,MA - EXP - ISSUE CHANGE THE PASSWORD FOR josym1,0.817136,Resolved,GENERIC
708,INC69280191,MA - EXP - NEW ACCOUNT ECIS LDAP josym1,0.803101,Resolved,GENERIC
718,INC69279802,MA - EXP - NEW ACCOUNT DCVS LDAP josym1,0.802676,Resolved,GENERIC
948,INC69215018,MA - EXP - DEMANDE D'ACCES AVAYA,0.780566,Resolved,GENERIC
66,INC70098069,MA - EXP - ESECURE ACCESS,0.771292,resolved,GENERIC


KeyError: 'Ranked_Clusters'

In [66]:
# ==========================================
# BACKTEST DE 4 TICKETS DE VALIDATION
# ==========================================

test_tickets = [
    "INC61302608",
    "INC65363499",
    "INC67571325",
    "INC72044971",
]

results = []

for ticket in test_tickets:

    print("=" * 70)
    print(f"BACKTEST : {ticket}")
    print("=" * 70)

    try:

        result = run_single_backtest(ticket)

        results.append({
            "Test_Ticket":
                result.get("Test_Ticket", ticket),

            "Status":
                result.get("status"),

            "Representative_Ticket":
                result.get("Representative_Ticket"),

            "Ticket_Similarity":
                result.get("Ticket_Similarity"),

            "Solution_Similarity":
                result.get("Solution_Similarity"),

            "Cluster_Score":
                result.get("Cluster_Score")
        })

        print("Status :", result.get("status"))
        print(
            "Ticket représentant :",
            result.get("Representative_Ticket")
        )
        print(
            "Ticket similarity :",
            result.get("Ticket_Similarity")
        )
        print(
            "Solution similarity :",
            result.get("Solution_Similarity")
        )
        print(
            "Cluster score :",
            result.get("Cluster_Score")
        )

    except Exception as e:

        print("ERREUR :", e)

        results.append({
            "Test_Ticket": ticket,
            "Status": "ERROR",
            "Representative_Ticket": None,
            "Ticket_Similarity": None,
            "Solution_Similarity": None,
            "Cluster_Score": None
        })


backtest_df = pd.DataFrame(results)

print("\n")
print("=" * 70)
print("RÉSUMÉ DU BACKTEST")
print("=" * 70)

display(backtest_df)

BACKTEST : INC61302608
Status : SUCCESS
Ticket représentant : INC69349039
Ticket similarity : 0.7695941925048828
Solution similarity : 0.8166919946670532
Cluster score : 0.8263850331306457
BACKTEST : INC65363499
Status : SUCCESS
Ticket représentant : INC61302608
Ticket similarity : 0.5952237844467163
Solution similarity : 0.6351664066314697
Cluster score : 0.7714965045452118
BACKTEST : INC67571325
Status : SUCCESS
Ticket représentant : INC67156306
Ticket similarity : 0.7628262042999268
Solution similarity : 0.3255992531776428
Cluster score : 0.7314131021499635
BACKTEST : INC72044971
Status : SUCCESS
Ticket représentant : INC61058067
Ticket similarity : 0.6624985933303833
Solution similarity : 0.1547107994556427
Cluster score : 0.6812492966651917


RÉSUMÉ DU BACKTEST


,Test_Ticket,Status,Representative_Ticket,Ticket_Similarity,Solution_Similarity,Cluster_Score
0,INC61302608,SUCCESS,INC69349039,0.769594,0.816692,0.826385
1,INC65363499,SUCCESS,INC61302608,0.595224,0.635166,0.771497
2,INC67571325,SUCCESS,INC67156306,0.762826,0.325599,0.731413
3,INC72044971,SUCCESS,INC61058067,0.662499,0.154711,0.681249


In [67]:
result = run_single_backtest(
    "INC67571325",
    top_k=50
)

display(
    result["All_Candidates"][
        [
            "Number",
            "Short Description",
            "Solution",
            "solution_type",
            "Ticket_Similarity"
        ]
    ]
)

,Number,Short Description,Solution,solution_type,Ticket_Similarity
707,INC69279844,MA - EXP - NEW ACCOUNT OCNT LDAP josym1,Resolved,GENERIC,0.844316
717,INC69280550,MA - EXP - NEW ACCOUNT DCE LDAP josym1,Access Granted,GENERIC,0.842690
703,INC69279873,MA - EXP - NEW ACCOUNT NMIV LDAP josym1,resolved,GENERIC,0.841746
706,INC69282830,MA - EXP - NEW ACCOUNT PRIM LDAP josym1,"Resolved , ticket ouvert INC69386410",NON_ACTIONABLE,0.838885
715,INC69282812,MA - EXP - NEW ACCOUNT HMS LDAP josym1,Resolved,GENERIC,0.831485
1492,INC68501709,MA - EXP - ISSUE CHANGE THE PASSWORD FOR josym1,Resolved,GENERIC,0.817136
708,INC69280191,MA - EXP - NEW ACCOUNT ECIS LDAP josym1,Resolved,GENERIC,0.803101
718,INC69279802,MA - EXP - NEW ACCOUNT DCVS LDAP josym1,Resolved,GENERIC,0.802676
948,INC69215018,MA - EXP - DEMANDE D'ACCES AVAYA,Resolved,GENERIC,0.780566
66,INC70098069,MA - EXP - ESECURE ACCESS,resolved,GENERIC,0.771292


In [64]:
diagnose_backtest("INC72044971")

DIAGNOSTIC : INC72044971

Status : NO_ACTIONABLE_SOLUTION

--- TICKET TESTÉ ---
Number : INC72044971
Short Description : MA - EXP - Correction of Phone Number on Invoices and Dunning Letters
Vraie solution : Fixed by Chetan

Aucun candidat trouvé.

BACKTEST NON COMPLET

Le pipeline n'a pas pu produire une recommandation finale.

Raison : NO_ACTIONABLE_SOLUTION

Aucune solution ACTIONABLE n'a été trouvée parmi les 10 meilleurs tickets similaires.

Répartition des types :
Series([], Name: count, dtype: int64)


{'status': 'NO_ACTIONABLE_SOLUTION',
 'Test_Ticket': 'INC72044971',
 'True_Solution': 'Fixed by Chetan',
 'All_Candidates':            Number              Opened Target Resolution Time (GMT)  \
 580   INC70547213 2026-06-05 12:44:39          2026-06-12 12:44:39   
 844   INC67220392 2025-12-04 10:22:53          2025-12-11 10:22:53   
 1609  INC71516608 2026-07-20 16:53:52          2026-07-27 16:53:52   
 1166  INC70939728 2026-06-24 09:22:51          2026-07-01 09:22:51   
 811   INC69585803 2026-04-16 09:53:42          2026-04-23 09:53:42   
 1519  INC69992578 2026-05-07 15:36:32          2026-05-14 15:36:32   
 986   INC66478390 2025-10-22 10:57:03          2025-10-29 11:57:03   
 1289  INC71939442 2026-08-06 09:21:19          2026-08-13 09:21:19   
 381   INC69288154 2026-03-31 15:45:37          2026-04-07 15:45:37   
 1788  INC70320505 2026-05-25 11:19:34          2026-06-01 11:19:34   
 
              Informed at Incident Status             Caller    Assigned Group  \
 580  2026-0

In [22]:
result = run_single_backtest(
    "INC61302608"
)

print("Status :", result["status"])
print("Ticket test :", result["Test_Ticket"])
print(
    "Ticket représentant :",
    result["Representative_Ticket"]
)
print(
    "Ticket similarity :",
    result["Ticket_Similarity"]
)
print(
    "Solution similarity :",
    result["Solution_Similarity"]
)
print(
    "Cluster score :",
    result["Cluster_Score"]
)

Status : SUCCESS
Ticket test : INC61302608
Ticket représentant : INC69407925
Ticket similarity : 0.7282114
Solution similarity : 0.09523153
Cluster score : 0.8379656195640564


In [30]:
# ==========================================
# DIAGNOSTIC COMPARATIF DES CANDIDATS
# ==========================================

def is_non_informative_solution(solution):
    """
    Détecte les solutions qui ne constituent pas
    une véritable action technique.
    """

    if pd.isna(solution):
        return True

    text = str(solution).strip().lower()

    if not text:
        return True

    # Solutions purement administratives
    non_informative = [
        "resolved",
        "resolve",
        "closed",
        "close",
        "done",
        "fixed"
    ]

    if text in non_informative:
        return True

    # Clôtures automatiques
    automatic_patterns = [
        "rule of 3",
        "rule of three",
        "set to status 'resolved'",
        "set to status resolved",
        "ticket was set to status"
    ]

    for pattern in automatic_patterns:
        if pattern in text:
            return True

    return False

In [31]:
# ==========================================
# DIAGNOSTIC SUR LES 4 TICKETS
# ==========================================

diagnostic_results = []

for ticket in test_tickets:

    result = run_single_backtest(ticket)

    candidates = result["Candidates"].copy()

    # Identifier les solutions non informatives
    candidates["Non_Informative"] = (
        candidates["Solution"]
        .apply(is_non_informative_solution)
    )

    # Meilleur candidat par similarité ticket
    best_candidate = candidates.iloc[0]

    # Meilleur candidat avec vraie solution
    informative_candidates = candidates[
        ~candidates["Non_Informative"]
    ]

    if not informative_candidates.empty:
        best_informative = informative_candidates.iloc[0]
    else:
        best_informative = None

    diagnostic_results.append({

        "Test_Ticket": ticket,

        "Pipeline_Representative":
            result.get("Representative_Ticket"),

        "Pipeline_Solution":
            result.get("Representative_Solution"),

        "Pipeline_Solution_Similarity":
            result.get("Solution_Similarity"),

        "Best_Ticket_Match":
            best_candidate["Number"],

        "Best_Ticket_Match_Similarity":
            best_candidate["Ticket_Similarity"],

        "Best_Ticket_Match_Solution":
            best_candidate["Solution"],

        "Best_Informative_Match":
            None if best_informative is None
            else best_informative["Number"],

        "Best_Informative_Similarity":
            None if best_informative is None
            else best_informative["Ticket_Similarity"],

        "Best_Informative_Solution":
            None if best_informative is None
            else best_informative["Solution"]
    })

diagnostic_df = pd.DataFrame(diagnostic_results)

diagnostic_df

,Test_Ticket,Pipeline_Representative,Pipeline_Solution,Pipeline_Solution_Similarity,Best_Ticket_Match,Best_Ticket_Match_Similarity,Best_Ticket_Match_Solution,Best_Informative_Match,Best_Informative_Similarity,Best_Informative_Solution
0,INC61302608,INC69407925,Resolved,0.095232,INC61302569,0.901350,"Completed, printers installed on demanded serv...",INC61302569,0.901350,"Completed, printers installed on demanded serv..."
1,INC65363499,INC61302569,"Completed, printers installed on demanded serv...",0.609194,INC65410555,0.863555,As per the request printer driver updated \n\n...,INC65410555,0.863555,As per the request printer driver updated \n\n...
2,INC67571325,INC69279844,Resolved,0.088936,INC69279844,0.844316,Resolved,INC69282830,0.838885,"Resolved , ticket ouvert INC69386410"
3,INC72044971,INC67220392,Resolved,0.453729,INC70547213,0.715061,resolved,INC69992578,0.689395,Resolved.


In [32]:
# ==========================================
# DIAGNOSTIC GLOBAL DES SOLUTIONS
# ==========================================

solution_stats = (
    knowledge_base
    .groupby(["source", "solution_type"])["Number"]
    .count()
    .reset_index(name="Count")
)

print(solution_stats)

      source   solution_type  Count
0  ESCALATED      ACTIONABLE    200
1  ESCALATED  NON_ACTIONABLE    230
2   RESOLVED      ACTIONABLE   1116
3   RESOLVED  NON_ACTIONABLE    697


In [33]:
# ==========================================
# SOLUTIONS LES PLUS FRÉQUENTES
# ==========================================

solution_frequency = (
    knowledge_base["Solution"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
    .value_counts()
)

solution_frequency.head(30)

Solution
resolved                                                                                                                                                                                                                                                                                                                   907
solved                                                                                                                                                                                                                                                                                                                     156
necessary done.                                                                                                                                                                                                                                                                                                             59
done                              

In [34]:
# ==========================================
# SOLUTIONS NON-INFORMATIVES
# ==========================================

knowledge_base["non_informative"] = (
    knowledge_base["Solution"]
    .apply(is_non_informative_solution)
)

print(
    knowledge_base["non_informative"]
    .value_counts()
)

print("\nExemples NON-INFORMATIFS :")

print(
    knowledge_base.loc[
        knowledge_base["non_informative"],
        ["Number", "Short Description", "Solution", "source"]
    ].head(20).to_string(index=False)
)

non_informative
False    1170
True     1073
Name: count, dtype: int64

Exemples NON-INFORMATIFS :
     Number                                                                                                  Short Description Solution   source
INC67542407                                                              MA EXP Application e-HCM OASYS not Working via Citrix   Closed RESOLVED
INC61393750                                                          MA EXP TUNSHIP Rrovide PID range for TN local application Resolved RESOLVED
INC64928937                                                                                                      DCE BAD ISSUE resolved RESOLVED
INC64305092                                                                            MA YYYYY Casablanca DCVS printing issue Resolved RESOLVED
INC61219128                                                                   MA - EXP - Probleme d'mpression et scan des docs Resolved RESOLVED
INC64864098 MA - EXP - le montan

In [36]:
knowledge_base["solution_type"] = (
    knowledge_base["Solution"]
    .apply(classify_solution)
)

print(knowledge_base["solution_type"].value_counts())

solution_type
GENERIC           1393
NON_ACTIONABLE     636
ACTIONABLE         178
ADMINISTRATIVE      36
Name: count, dtype: int64


In [37]:
result = run_single_backtest("INC61302608")

In [38]:
result = run_single_backtest("INC61302608")

print(result)

{'status': 'SUCCESS', 'Test_Ticket': 'INC61302608', 'True_Solution': 'done all mentioned printers installed on servers.\n\nMYKULLSPC000578\ndevice for MATNGKEN03: beh:/1/0/60/socket://23.82.61.234:9100\nMATNGKEN03 accepting requests since Thu 17 Oct 2024 08:10:21 AM GMT\nprinter MATNGKEN03 is idle.  enabled since Thu 17 Oct 2024 08:10:21 AM GMT\ndevice for MATNGKEN04: beh:/1/0/60/socket://23.82.61.235:9100\nMATNGKEN04 accepting requests since Thu 17 Oct 2024 08:10:21 AM GMT\nprinter MATNGKEN04 is idle.  enabled since Thu 17 Oct 2024 08:10:21 AM GMT\ndevice for MARBARAK03: beh:/1/0/60/socket://23.82.50.234:9100\nMARBARAK03 accepting requests since Thu 17 Oct 2024 08:10:21 AM GMT\nprinter MARBARAK03 is idle.  enabled since Thu 17 Oct 2024 08:10:21 AM GMT\ndevice for MARBARAK04: beh:/1/0/60/socket://23.82.50.235:9100\nMARBARAK04 accepting requests since Thu 17 Oct 2024 08:10:21 AM GMT\nprinter MARBARAK04 is idle.  enabled since Thu 17 Oct 2024 08:10:21 AM GMT\ndevice for MARBAAGA03: beh:/

In [39]:
print(knowledge_base["solution_type"].value_counts(dropna=False))
print("\nValeurs présentes :")
print(knowledge_base["solution_type"].unique())

solution_type
GENERIC           1393
NON_ACTIONABLE     636
ACTIONABLE         178
ADMINISTRATIVE      36
Name: count, dtype: int64

Valeurs présentes :
<StringArray>
['NON_ACTIONABLE', 'GENERIC', 'ADMINISTRATIVE', 'ACTIONABLE']
Length: 4, dtype: str


In [40]:
classes = [
    "ACTIONABLE",
    "GENERIC",
    "ADMINISTRATIVE",
    "NON_ACTIONABLE"
]

for cls in classes:
    subset = knowledge_base[
        knowledge_base["solution_type"] == cls
    ][["Number", "Short Description", "Solution"]]

    print(f"\n{'='*80}")
    print(f"### {cls} — {len(subset)} tickets")
    print(f"{'='*80}")

    if len(subset) > 0:
        display(
            subset.sample(
                min(20, len(subset)),
                random_state=42
            )
        )


### ACTIONABLE — 178 tickets


,Number,Short Description,Solution
980,INC66989858,MA - EXP - SAFENET,installed
1654,INC69043723,MA - EXP - MA dashboard access DCVS,I inform you that the ticket has been created ...
2042,INC68309155,Morocco Daylight Saving,all the mentioned applications are updated the...
1389,INC66615978,MA - EXP - Installation imprimante sur PC airops,Installation done
1865,INC62972826,MA ACCS skillsets Start/End hour change during...,Working hours for all departments have been ch...
891,INC68608600,MA - EXP - PowerBI Desktop Application install...,L'installation seras faite des mon retour de l...
1984,INC67693039,Inquiry Regarding Auto Manifest FTP Integratio...,Updated SFTP details as requested.
2235,INC67708996,EXP-MA : Request to change Campfire MA Admin.,Updates morroco country admins as per the requ...
1964,INC61363315,Change IP ADR,The printers ip address has been changed as re...
2043,INC71376046,Remove users from DCC,"Dear Nabil, \n\nplease raise the request for d..."



### GENERIC — 1393 tickets


,Number,Short Description,Solution
429,INC68475690,MA - EXP - CENTRALE GYPSE,Necessary done.
1448,INC67544628,Problème deconnexion des caméras Stations,solved
1276,INC62293076,MA - EXP - no bip,Resolved
683,INC68355502,MA - EXP - Acces to PMT,Closed
828,INC70126083,MA - EXP - e-secure admin,RESOLVED
737,INC70496770,MA - EXP - Ajouter cette adresse : sales7@go2s...,Resolved
1771,INC62517339,MA - EXP - problem displaying the chat bar,Resolved
350,INC69660635,MA - EXP - FICHIER WORCK PORTNET,Necessary done.
544,INC67391156,MA - EXP - blocage pc RW,Resolved
73,INC70521522,MA - EXP - Mydhl+ Esecure Admin,resolved



### ADMINISTRATIVE — 36 tickets


,Number,Short Description,Solution
2085,INC69345776,Needs OCNT worstations & printers details,i proceed to close the ticket
1179,INC66991194,MCT MAIL TO REMOVE,The process is ongoing and will take some time...
1641,INC69148728,MA - EXP - Harmonization of incoming calls num...,"Hi Wafaa, the new ticket has been forwarded to..."
1911,INC64953151,"MA - EXP - Dashboard CSV Remote Booking ""Escal...","issue:\nDashboard CSV Remote Booking ""Escalati..."
1328,INC64880629,"MA - EXP - Dashboard CSV Remote Booking ""Escal...",regional ticket opened
1968,INC66134244,MA EXP KIWI Request for information regarding...,"As confirmed over teams chat, we are proceedin..."
1468,INC69314198,Needs list of worstations (Country : MA),A new ticket has been raised
1177,INC64882013,"MA - EXP - GEMA / QUICK ACTION ""CALL BACK ESCA...",regional ticket opened
904,INC71096044,MA - EXP - Demande rapport GSA comptes Cash,Data provided
1339,INC64879657,MA - EXP - Chat working Hour,Regional ticket opened



### NON_ACTIONABLE — 636 tickets


,Number,Short Description,Solution
1368,INC68357270,MA - EXP - TV ISSUE,"Hello Mariam,\nITs not our responsability. you..."
2234,INC67184961,MA EXP SABLE login Reset,**************************** SOLUTION - READ M...
346,INC67690184,MA - EXP - Esecure,"Hi Ilyass,\n\nAs already communicated , starti..."
1825,INC66030605,Restore GSN roles for user : yozmirli (youness...,We have provided the ITIL role to youness.zmir...
711,INC63870826,MA - EXP - Issue print quality of zebra printer,"Salam Oussama, comme convenu au téléphone, l'e..."
2192,INC64107172,Avaya extension,as discussed and troubleshooted by Yassir on T...
239,INC66236754,CDC: MA EXP - User account: prgdca-xnet-svpn01...,The user had some issues to get connected yest...
365,INC71678895,"MA - EXP - dear team, could you please grant ...",INC71770314 ticket ouvert
2163,INC63027190,MA EXP VISTA Access request,"Dear colleague, because the required informati..."
1025,INC68607375,MA - EXP - Probleme des nouveaux Carte SIM orange,Issue under Provider investigations


In [41]:
nonact = knowledge_base[
    knowledge_base["solution_type"] == "NON_ACTIONABLE"
][["Number", "Short Description", "Solution"]]

display(
    nonact.sample(
        min(50, len(nonact)),
        random_state=42
    )
)

,Number,Short Description,Solution
1368,INC68357270,MA - EXP - TV ISSUE,"Hello Mariam,\nITs not our responsability. you..."
2234,INC67184961,MA EXP SABLE login Reset,**************************** SOLUTION - READ M...
346,INC67690184,MA - EXP - Esecure,"Hi Ilyass,\n\nAs already communicated , starti..."
1825,INC66030605,Restore GSN roles for user : yozmirli (youness...,We have provided the ITIL role to youness.zmir...
711,INC63870826,MA - EXP - Issue print quality of zebra printer,"Salam Oussama, comme convenu au téléphone, l'e..."
2192,INC64107172,Avaya extension,as discussed and troubleshooted by Yassir on T...
239,INC66236754,CDC: MA EXP - User account: prgdca-xnet-svpn01...,The user had some issues to get connected yest...
365,INC71678895,"MA - EXP - dear team, could you please grant ...",INC71770314 ticket ouvert
2163,INC63027190,MA EXP VISTA Access request,"Dear colleague, because the required informati..."
1025,INC68607375,MA - EXP - Probleme des nouveaux Carte SIM orange,Issue under Provider investigations


In [42]:
print("Nombre NON_ACTIONABLE :", len(nonact))

print("\nSolutions les plus fréquentes :")
print(
    nonact["Solution"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
    .value_counts()
    .head(30)
)

Nombre NON_ACTIONABLE : 636

Solutions les plus fréquentes :
Solution
dear colleague, because the required information needed to resolve your inc ticket was not provided by you, this ticket was set to status 'resolved' by the rule of 3 automation in accordance with the incident management process. however, if your issue persists, please contact your home service desk. thank you.    45
customer status is now active. a temporary password has been sent to the customer's email address.                                                                                                                                                                                                                          5
enable notifications and sound                                                                                                                                                                                                                                                                        

In [43]:
# Solutions NON_ACTIONABLE qui semblent contenir une action réelle
action_words = [
    "created",
    "create",
    "granted",
    "grant",
    "installed",
    "installation",
    "updated",
    "update",
    "changed",
    "change",
    "removed",
    "remove",
    "reset",
    "unlocked",
    "unlock",
    "enabled",
    "enable",
    "disabled",
    "disable",
    "configured",
    "configuration",
    "recreated",
    "replaced",
    "fixed",
    "released",
    "restored",
    "provided",
    "completed",
    "activated",
    "activated",
    "password",
    "access"
]

mask = (
    knowledge_base["solution_type"].eq("NON_ACTIONABLE")
    & knowledge_base["Solution"].fillna("").str.lower().apply(
        lambda x: any(word in x for word in action_words)
    )
)

potential_actionable = knowledge_base.loc[
    mask,
    ["Number", "Short Description", "Solution"]
]

print("NON_ACTIONABLE potentiellement ACTIONABLE :", len(potential_actionable))

display(
    potential_actionable.sample(
        min(50, len(potential_actionable)),
        random_state=42
    )
)

NON_ACTIONABLE potentiellement ACTIONABLE : 257


,Number,Short Description,Solution
2049,INC69064633,MA EXP KIWI Request for information regarding...,"Dear colleague, because the required informati..."
2100,INC69072848,MA ACCS skillsets duty Start/End time on Monda...,completed
1464,INC69678782,Changement customs file / AKSAL,Changes ongoing
439,INC68477217,MA - EXP - Please verify the mail : M.MEJBAR@A...,Customer status is now active. A temporary pas...
223,INC67495185,MA - EXP - ADDING EMAIL AS ADMINISTRATOR FOR M...,The Necessary changes are made.
163,INC67125997,MA - EXP - Connection Slowdowns and Delayed Ch...,Fixed.\nIssue was due to the slow connection
2138,INC64093094,MA EXP NMIV Access,**************************** SOLUTION - READ M...
339,INC68542207,MA - EXP - To change the OLD Admin Contact fro...,Necessary changes are made.
1915,INC70618611,Error access Basware web application - dhlexpr...,"Hi, \n\nWe faced system down and Basware Suppo..."
1856,INC66525504,Slowness issue in MA SDWAN site : EMTO-MA-EXP-002,BT Time of Resolution : 2025-11-03T11:27:53Z\n...


In [68]:
# ==========================================
# CONFIGURATION FINALE DU BACKTEST
# ==========================================

import numpy as np
import pandas as pd
import re

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import AgglomerativeClustering


# ==========================================
# CLASSIFICATION DES SOLUTIONS
# ==========================================

def classify_solution(solution):

    if pd.isna(solution):
        return "NON_ACTIONABLE"

    s = str(solution).strip().lower()

    if not s:
        return "NON_ACTIONABLE"

    # ======================================
    # ADMINISTRATIVE
    # ======================================

    administrative_patterns = [

        "duplicate",
        "duplicated",
        "close the ticket",
        "ticket to avoid",
        "no response from user",
        "regional ticket opened",
        "new ticket has been raised",
        "a new ticket has been raised",
        "ticket has been raised",
        "ticket opened",
        "data provided",
        "information provided",
        "wrongly routed",
        "cancelled",
        "cancelled by"
    ]

    if any(
        pattern in s
        for pattern in administrative_patterns
    ):
        return "ADMINISTRATIVE"

    # ======================================
    # NON ACTIONABLE
    # ======================================

    non_actionable_patterns = [

        "issue under provider investigations",
        "provider is working on it",
        "provider is working on the issue",
        "no approval is provided",
        "required information needed to resolve",
        "required information was not provided",
        "please ask performance to provide",
        "please reach out to",
        "please contact",
        "please request access from",
        "not our responsibility",
        "no issue from our side",
        "waiting for",
        "waiting on",
        "pending",
        "under investigation",
        "changes ongoing",
        "will be changed",
        "will be configured",
        "will be installed"
    ]

    if any(
        pattern in s
        for pattern in non_actionable_patterns
    ):
        return "NON_ACTIONABLE"

    # ======================================
    # GENERIC
    # ======================================

    generic_solutions = {

        "resolved",
        "resolved.",
        "resolved,",

        "solved",
        "solved.",

        "done",
        "done.",

        "fixed",
        "fixed.",

        "closed",
        "closed.",

        "completed",
        "completed.",

        "granted",

        "access granted",
        "access granted.",

        "necessary done",
        "necessary done.",

        "necessaey done.",

        "necessary changes are made.",

        "the necessary changes are made.",

        "change done",
        "change done.",

        "fixed1"
    }

    if s in generic_solutions:
        return "GENERIC"

    if re.match(
        r"^(resolved|fixed)\s+inc[\w-]+",
        s
    ):
        return "GENERIC"

    # ======================================
    # ACTIONABLE
    # ======================================

    actionable_patterns = [

        "installed",
        "installation done",

        "created",
        "was created",
        "has been created",

        "configured",
        "configuration done",

        "updated",
        "has been updated",
        "was updated",

        "changed",
        "has been changed",
        "was changed",

        "replaced",
        "has been replaced",

        "removed",
        "has been removed",

        "reset",

        "password generated",
        "password has been sent",
        "password was sent",

        "unlocked",
        "unlock account",

        "access provided",

        "granted the",

        "enabled",
        "disabled",

        "recreated",

        "fixed by",

        "released",
        "session released",

        "restored",

        "activated",

        "completed successfully",

        "request completed",

        "changes complete",

        "changes were made",

        "changes are made",

        "set as requested",

        "printer has been created",

        "printers were recreated",

        "rebooted",

        "reconfigured",

        "configured a new",

        "synced",

        "provided the access",

        "has been granted"
    ]

    if any(
        pattern in s
        for pattern in actionable_patterns
    ):
        return "ACTIONABLE"

    return "NON_ACTIONABLE"


# ==========================================
# APPLICATION DE LA CLASSIFICATION
# ==========================================

knowledge_base["solution_type"] = (

    knowledge_base["Solution"]

    .apply(
        classify_solution
    )
)


# ==========================================
# INFORMATION
# ==========================================

print(
    knowledge_base[
        "solution_type"
    ].value_counts()
)

solution_type
GENERIC           1401
NON_ACTIONABLE     599
ACTIONABLE         201
ADMINISTRATIVE      42
Name: count, dtype: int64


In [69]:
# ==========================================
# FONCTIONS FINALES DE CLUSTERING
# ==========================================


def cluster_similar_solutions(
    solution_similarity_matrix,
    threshold=0.75
):
    """
    Regroupe les solutions similaires.
    """

    # ======================================
    # Aucun candidat
    # ======================================

    if solution_similarity_matrix.empty:

        return pd.DataFrame(
            columns=[
                "Number",
                "Cluster"
            ]
        )

    # ======================================
    # Un seul candidat
    # ======================================

    if len(
        solution_similarity_matrix
    ) == 1:

        return pd.DataFrame({

            "Number":
                solution_similarity_matrix.index,

            "Cluster":
                [0]
        })

    # ======================================
    # Matrice de distance
    # ======================================

    distance_matrix = (

        1
        -
        solution_similarity_matrix.values
    )

    # ======================================
    # Clustering
    # ======================================

    clustering = AgglomerativeClustering(

        n_clusters=None,

        metric="precomputed",

        linkage="average",

        distance_threshold=(
            1 - threshold
        )
    )

    labels = clustering.fit_predict(
        distance_matrix
    )

    # ======================================
    # Résultat
    # ======================================

    return pd.DataFrame({

        "Number":
            solution_similarity_matrix.index,

        "Cluster":
            labels
    })


# ==========================================
# SCORING DES CLUSTERS
# ==========================================

def score_solution_clusters(
    actionable,
    clusters,
    solution_similarity_matrix
):

    cluster_results = []

    if clusters.empty:

        return pd.DataFrame()

    for cluster_id in sorted(

        clusters[
            "Cluster"
        ].unique()

    ):

        cluster_numbers = (

            clusters.loc[

                clusters["Cluster"]
                == cluster_id,

                "Number"

            ].tolist()
        )

        cluster_data = (

            actionable[

                actionable["Number"].isin(
                    cluster_numbers
                )

            ].copy()
        )

        if cluster_data.empty:
            continue

        # ==================================
        # Pertinence moyenne
        # ==================================

        ticket_similarity_mean = (

            cluster_data[
                "Ticket_Similarity"
            ].mean()
        )

        # ==================================
        # Pertinence maximale
        # ==================================

        ticket_similarity_max = (

            cluster_data[
                "Ticket_Similarity"
            ].max()
        )

        # ==================================
        # Cohérence des solutions
        # ==================================

        valid_tickets = (

            cluster_data[
                "Number"
            ].tolist()
        )

        if len(valid_tickets) > 1:

            matrix = (

                solution_similarity_matrix.loc[

                    valid_tickets,

                    valid_tickets

                ].values
            )

            upper_values = matrix[

                np.triu_indices(

                    len(matrix),

                    k=1

                )
            ]

            solution_coherence = float(

                upper_values.mean()

            )

        else:

            # Singleton :
            # pas assez d'évidence historique

            solution_coherence = 0.50

        # ==================================
        # Résultat
        # ==================================

        cluster_results.append({

            "Cluster":
                cluster_id,

            "Number_of_Tickets":
                len(cluster_data),

            "Ticket_Similarity_Mean":
                ticket_similarity_mean,

            "Ticket_Similarity_Max":
                ticket_similarity_max,

            "Solution_Coherence":
                solution_coherence
        })

    return pd.DataFrame(
        cluster_results
    )


# ==========================================
# CLASSEMENT DES CLUSTERS
# ==========================================

def rank_solution_clusters(
    cluster_scores
):

    if cluster_scores.empty:

        return cluster_scores.copy()

    scores = cluster_scores.copy()

    # ======================================
    # RELEVANCE
    # ======================================

    scores["Relevance_Score"] = (

        scores[
            "Ticket_Similarity_Mean"
        ]
    )

    # ======================================
    # COHERENCE
    # ======================================

    scores["Coherence_Score"] = (

        scores[
            "Solution_Coherence"
        ]
    )

    # ======================================
    # SUPPORT
    # ======================================

    max_support = (

        scores[
            "Number_of_Tickets"
        ].max()
    )

    scores["Support_Score"] = np.where(

        scores[
            "Number_of_Tickets"
        ] >= 2,

        np.log1p(

            scores[
                "Number_of_Tickets"
            ]

        )

        /

        np.log1p(
            max_support
        ),

        0.0
    )

    # ======================================
    # SCORE FINAL
    # ======================================

    scores[
        "Cluster_Final_Score"
    ] = (

        0.50
        *
        scores[
            "Relevance_Score"
        ]

        +

        0.30
        *
        scores[
            "Coherence_Score"
        ]

        +

        0.20
        *
        scores[
            "Support_Score"
        ]
    )

    # ======================================
    # TRI
    # ======================================

    return (

        scores

        .sort_values(

            "Cluster_Final_Score",

            ascending=False

        )

        .reset_index(
            drop=True
        )
    )

In [70]:
# ==========================================
# BACKTEST FINAL
# ==========================================

def run_single_backtest(

    test_ticket_number,

    top_k=50,

    cluster_threshold=0.75

):
    """
    Évalue le pipeline sur un ticket historique.

    Le ticket testé est retiré de la knowledge base
    pour éviter toute fuite de données.

    Le backtest retourne une recommandation seulement
    lorsqu'il existe suffisamment d'évidence historique.
    """

    # ======================================
    # 1. LOCALISER LE TICKET TEST
    # ======================================

    matches = knowledge_base.index[

        knowledge_base["Number"]

        .astype(str)

        .eq(
            str(test_ticket_number)
        )

    ].tolist()

    if len(matches) == 0:

        raise ValueError(

            f"Ticket introuvable : "
            f"{test_ticket_number}"
        )

    if len(matches) > 1:

        raise ValueError(

            f"Plusieurs tickets trouvés : "
            f"{test_ticket_number}"
        )

    test_index = matches[0]

    test_row = knowledge_base.loc[
        test_index
    ]

    true_solution = str(
        test_row["Solution"]
    )

    # ======================================
    # 2. RETIRER LE TICKET TEST
    # ======================================

    local_indices = knowledge_base.index[

        knowledge_base.index
        != test_index

    ].to_numpy()

    local_kb = knowledge_base.loc[
        local_indices
    ].copy()

    local_ticket_embeddings = (

        embeddings[
            local_indices
        ]
    )

    # ======================================
    # 3. EMBEDDING DU TICKET TEST
    # ======================================

    query_embedding = (

        embeddings[
            test_index
        ]
    )

    # =========ok =============================
    # 4. RECHERCHE DES TICKETS SIMILAIRES
    # ======================================

    ticket_similarities = (

        cosine_similarity(

            local_ticket_embeddings,

            query_embedding.reshape(
                1,
                -1
            )

        ).flatten()
    )

    top_k = min(
        top_k,
        len(local_kb)
    )

    top_positions = (

        np.argsort(
            ticket_similarities
        )[::-1][:top_k]
    )

    all_candidates = (

        local_kb.iloc[
            top_positions
        ].copy()
    )

    all_candidates[
        "Ticket_Similarity"
    ] = (

        ticket_similarities[
            top_positions
        ]
    )

    all_candidates = (

        all_candidates

        .sort_values(

            "Ticket_Similarity",

            ascending=False
        )

        .copy()
    )

    # ======================================
    # 5. FILTRAGE ACTIONABLE
    # ======================================

    actionable = (

        all_candidates[

            all_candidates[
                "solution_type"
            ]

            == "ACTIONABLE"

        ].copy()
    )

    # ======================================
    # AUCUNE SOLUTION ACTIONABLE
    # ======================================

    if actionable.empty:

        return {

            "status":
                "NO_ACTIONABLE_SOLUTION",

            "Test_Ticket":
                test_ticket_number,

            "True_Solution":
                true_solution,

            "All_Candidates":
                all_candidates,

            "Candidates":
                actionable,

            "Confidence_Score":
                0.0,

            "Confidence_Level":
                "LOW"
        }

    # ======================================
    # 6. EMBEDDINGS DES SOLUTIONS
    # ======================================

    candidate_original_indices = (

        actionable.index.to_numpy()
    )

    candidate_solution_embeddings = (

        solution_embeddings[
            candidate_original_indices
        ]
    )

    # ======================================
    # 7. MATRICE DE SIMILARITÉ
    # ======================================

    if len(actionable) == 1:

        solution_similarity_matrix = (

            pd.DataFrame(

                [[1.0]],

                index=
                    actionable["Number"],

                columns=
                    actionable["Number"]
            )
        )

    else:

        solution_matrix = (

            cosine_similarity(
                candidate_solution_embeddings
            )
        )

        solution_similarity_matrix = (

            pd.DataFrame(

                solution_matrix,

                index=
                    actionable["Number"],

                columns=
                    actionable["Number"]
            )
        )

    # ======================================
    # 8. CLUSTERING
    # ======================================

    clusters = (

        cluster_similar_solutions(

            solution_similarity_matrix,

            threshold=
                cluster_threshold
        )
    )

    # ======================================
    # 9. SCORING
    # ======================================

    cluster_scores = (

        score_solution_clusters(

            actionable,

            clusters,

            solution_similarity_matrix
        )
    )

    if cluster_scores.empty:

        return {

            "status":
                "NO_VALID_CLUSTER",

            "Test_Ticket":
                test_ticket_number,

            "True_Solution":
                true_solution,

            "All_Candidates":
                all_candidates,

            "Candidates":
                actionable,

            "Confidence_Score":
                0.0,

            "Confidence_Level":
                "LOW"
        }

    # ======================================
    # 10. CLASSEMENT
    # ======================================

    ranked_clusters = (

        rank_solution_clusters(
            cluster_scores
        )
    )

    # ======================================
    # 11. MEILLEUR CLUSTER
    # ======================================

    best_cluster_data = (

        ranked_clusters.iloc[0]
    )

    best_cluster = int(

        best_cluster_data[
            "Cluster"
        ]
    )

    number_of_tickets = int(

        best_cluster_data[
            "Number_of_Tickets"
        ]
    )

    # ======================================
    # 12. TICKETS DU CLUSTER
    # ======================================

    cluster_numbers = (

        clusters.loc[

            clusters["Cluster"]
            == best_cluster,

            "Number"

        ].tolist()
    )

    cluster_data = (

        actionable[

            actionable["Number"].isin(
                cluster_numbers
            )

        ].copy()
    )

    # ======================================
    # 13. TICKET REPRÉSENTATIF
    # ======================================

    if len(cluster_numbers) == 1:

        representative = (
            cluster_data.iloc[0]
        )

        representative_ticket = (

            representative["Number"]
        )

        representative_solution = str(

            representative["Solution"]
        )

        internal_similarity = 0.50

    else:

        matrix = (

            solution_similarity_matrix.loc[

                cluster_numbers,

                cluster_numbers
            ]
        )

        mean_similarities = {}

        for ticket_number in cluster_numbers:

            others = [

                number

                for number
                in cluster_numbers

                if number
                != ticket_number
            ]

            mean_similarities[
                ticket_number
            ] = (

                matrix.loc[

                    ticket_number,

                    others

                ].mean()
            )

        representative_ticket = max(

            mean_similarities,

            key=
                mean_similarities.get
        )

        representative = (

            cluster_data[

                cluster_data["Number"]
                == representative_ticket

            ].iloc[0]
        )

        representative_solution = str(

            representative["Solution"]
        )

        internal_similarity = float(

            mean_similarities[
                representative_ticket
            ]
        )

    # ======================================
    # 14. SCORES
    # ======================================

    ticket_similarity = float(

        representative[
            "Ticket_Similarity"
        ]
    )

    cluster_score = float(

        best_cluster_data[
            "Cluster_Final_Score"
        ]
    )

    # ======================================
    # 15. CONFIDENCE SCORE
    # ======================================

    confidence_score = (

        0.50
        *
        ticket_similarity

        +

        0.30
        *
        internal_similarity

        +

        0.20
        *
        cluster_score
    )

    # ======================================
    # PÉNALITÉ SINGLETON
    # ======================================

    if number_of_tickets == 1:

        confidence_score *= 0.60

    # ======================================
    # 16. CONFIDENCE LEVEL
    # ======================================

    if (

        confidence_score >= 0.80

        and

        number_of_tickets >= 2

    ):

        confidence_level = "HIGH"

    elif (

        confidence_score >= 0.60

        and

        number_of_tickets >= 2

    ):

        confidence_level = "MEDIUM"

    else:

        confidence_level = "LOW"

    # ======================================
    # 17. SIMILARITÉ AVEC VRAIE SOLUTION
    #
    # UNIQUEMENT POUR ÉVALUATION
    # ======================================

    true_solution_embedding = (

        solution_embeddings[
            test_index
        ]
    )

    representative_index = (

        actionable[

            actionable["Number"]

            .astype(str)

            .eq(
                str(
                    representative_ticket
                )
            )

        ].index[0]
    )

    representative_solution_embedding = (

        solution_embeddings[
            representative_index
        ]
    )

    solution_similarity = float(

        cosine_similarity(

            [
                true_solution_embedding
            ],

            [
                representative_solution_embedding
            ]

        )[0][0]
    )

    # ======================================
    # 18. DÉCISION FINALE
    # ======================================

    # --------------------------------------
    # Cas 1 :
    #
    # Cluster singleton
    # + faible confiance
    #
    # On refuse de présenter la solution
    # comme fiable.
    # --------------------------------------

    if (

        number_of_tickets == 1

        and

        confidence_score < 0.60

    ):

        status = "LOW_CONFIDENCE"

    else:

        status = "SUCCESS"

    # ======================================
    # 19. RÉSULTAT FINAL
    # ======================================

    return {

        "status":
            status,

        "Test_Ticket":
            test_ticket_number,

        "True_Solution":
            true_solution,

        "Best_Cluster":
            best_cluster,

        "Number_of_Tickets":
            number_of_tickets,

        "Representative_Ticket":
            representative_ticket,

        "Representative_Solution":
            representative_solution,

        "Ticket_Similarity":
            ticket_similarity,

        "Internal_Similarity":
            internal_similarity,

        "Cluster_Score":
            cluster_score,

        "Solution_Similarity":
            solution_similarity,

        "Confidence_Score":
            confidence_score,

        "Confidence_Level":
            confidence_level,

        "All_Candidates":
            all_candidates,

        "Candidates":
            actionable,

        "Ranked_Clusters":
            ranked_clusters
    }

In [71]:
# ==========================================
# BACKTEST FINAL SUR 4 TICKETS
# ==========================================

test_tickets = [

    "INC61302608",

    "INC65363499",

    "INC67571325",

    "INC72044971"
]


results = []


for ticket in test_tickets:

    print("=" * 70)

    print(
        f"BACKTEST : {ticket}"
    )

    print("=" * 70)

    try:

        result = run_single_backtest(

            ticket,

            top_k=50
        )

        results.append({

            "Test_Ticket":
                result.get(
                    "Test_Ticket"
                ),

            "Status":
                result.get(
                    "status"
                ),

            "Representative_Ticket":
                result.get(
                    "Representative_Ticket"
                ),

            "Number_of_Tickets":
                result.get(
                    "Number_of_Tickets"
                ),

            "Ticket_Similarity":
                result.get(
                    "Ticket_Similarity"
                ),

            "Internal_Similarity":
                result.get(
                    "Internal_Similarity"
                ),

            "Cluster_Score":
                result.get(
                    "Cluster_Score"
                ),

            "Solution_Similarity":
                result.get(
                    "Solution_Similarity"
                ),

            "Confidence_Score":
                result.get(
                    "Confidence_Score"
                ),

            "Confidence_Level":
                result.get(
                    "Confidence_Level"
                )
        })


        print()

        print(
            "Status :",
            result.get("status")
        )

        print(
            "Représentant :",
            result.get(
                "Representative_Ticket"
            )
        )

        print(
            "Nombre tickets cluster :",
            result.get(
                "Number_of_Tickets"
            )
        )

        print(
            "Ticket similarity :",
            result.get(
                "Ticket_Similarity"
            )
        )

        print(
            "Solution similarity :",
            result.get(
                "Solution_Similarity"
            )
        )

        print(
            "Confidence score :",
            result.get(
                "Confidence_Score"
            )
        )

        print(
            "Confidence level :",
            result.get(
                "Confidence_Level"
            )
        )


    except Exception as e:

        print(
            "ERREUR :",
            e
        )

        results.append({

            "Test_Ticket":
                ticket,

            "Status":
                "ERROR"
        })


# ==========================================
# TABLEAU FINAL
# ==========================================

backtest_df = pd.DataFrame(
    results
)

print()

print("=" * 70)

print(
    "RÉSUMÉ FINAL DU BACKTEST"
)

print("=" * 70)


display(backtest_df)

BACKTEST : INC61302608

Status : SUCCESS
Représentant : INC69349039
Nombre tickets cluster : 3
Ticket similarity : 0.7695941925048828
Solution similarity : 0.8166919946670532
Confidence score : 0.7885710144042968
Confidence level : MEDIUM
BACKTEST : INC65363499

Status : SUCCESS
Représentant : INC61302608
Nombre tickets cluster : 5
Ticket similarity : 0.5952237844467163
Solution similarity : 0.6351664066314697
Confidence score : 0.7109375178813935
Confidence level : MEDIUM
BACKTEST : INC67571325

Status : LOW_CONFIDENCE
Représentant : INC67156306
Nombre tickets cluster : 1
Ticket similarity : 0.7628262042999268
Solution similarity : 0.3255992531776428
Confidence score : 0.3826174335479737
Confidence level : LOW
BACKTEST : INC72044971

Status : LOW_CONFIDENCE
Représentant : INC61058067
Nombre tickets cluster : 1
Ticket similarity : 0.6624985933303833
Solution similarity : 0.1547107994556427
Confidence score : 0.346499493598938
Confidence level : LOW

RÉSUMÉ FINAL DU BACKTEST


,Test_Ticket,Status,Representative_Ticket,Number_of_Tickets,Ticket_Similarity,Internal_Similarity,Cluster_Score,Solution_Similarity,Confidence_Score,Confidence_Level
0,INC61302608,SUCCESS,INC69349039,3,0.769594,0.794990,0.826385,0.816692,0.788571,MEDIUM
1,INC65363499,SUCCESS,INC61302608,5,0.595224,0.863421,0.771497,0.635166,0.710938,MEDIUM
2,INC67571325,LOW_CONFIDENCE,INC67156306,1,0.762826,0.500000,0.531413,0.325599,0.382617,LOW
3,INC72044971,LOW_CONFIDENCE,INC61058067,1,0.662499,0.500000,0.481249,0.154711,0.346499,LOW
